In [1]:
from pathlib import Path

import prism

from imagematerials.eol import eol_preprocess
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    EndOfLife,
    GenericMaterials,
    GenericStocks,
    Maintenance,
    MaterialIntensities,
    RestOf
)
from imagematerials.preprocessing import get_preprocessing_data
import numpy as np

from imagematerials.rest_of import rest_of_preprocessing
import matplotlib.pyplot as plt

from imagematerials.electricity.preprocessing import (
    get_preprocessing_data_gen,
    get_preprocessing_data_grid)


In [2]:
scenario_name = "SSP2_M_CP"

In [3]:
climate_policy_scenario_dir = Path("..", "data", "raw", "image", scenario_name)
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'

path_image_materials = Path("C:/Coding/image-materials")

raw_data = path_image_materials / "data/raw"

In [4]:
#load historic data
from imagematerials.rest_of.resource_model import ResourceModel

steel = ResourceModel(resource_group = 'metals', resource = 'steel', 
                        image_mat_available = True, start_year = 1971,
                        scenario=scenario_name,
                        convert_image=True, end_year = 2012, convert_to_tons = 1/1000_000, 
                        trade_data=True, path_input_data="../data/raw/rest-of", 
                        path_input_data_image = Path("../data/raw/image"))

aluminium = ResourceModel(resource_group = 'metals', resource = 'aluminium', 
                        image_mat_available = True, start_year = 1998, 
                        scenario=scenario_name, end_year = 2024, path_input_data="../data/raw/rest-of", 
                        path_input_data_image = Path("../data/raw/image")
                        )

cement = ResourceModel(resource_group = 'nmm', resource = 'cement', 
                    image_mat_available = True, start_year = 1971, 
                    scenario=scenario_name,
                    convert_image=True, end_year = 2012, convert_to_tons = 1/1000_000, 
                    trade_data=True, path_input_data="../data/raw/rest-of", 
                    path_input_data_image = Path("../data/raw/image"))

copper = ResourceModel(resource_group = 'metals', resource = 'copper', 
                       image_mat_available = True, start_year = 1990,
                       scenario= scenario_name, end_year = 2011,
                       path_input_data="../data/raw/rest-of", 
                       path_input_data_image = Path("../data/raw/image"))

sand = ResourceModel(resource_group = 'nmm', resource = 'sand_gravel_crushed_rock', 
                       image_mat_available = True, start_year = 1970, scenario= scenario_name, 
                       path_input_data="../data/raw/rest-of", 
                    path_input_data_image = Path("../data/raw/image"))

limestone = ResourceModel(resource_group = 'nmm', resource = 'limestone', 
                    image_mat_available = False, start_year = 1970, scenario=scenario_name, 
                    path_input_data="../data/raw/rest-of", 
                    path_input_data_image = Path("../data/raw/image"))

clay = ResourceModel(resource_group = 'nmm', resource = 'clays', 
                    image_mat_available = False, start_year = 1970, 
                    scenario=scenario_name, 
                    path_input_data="../data/raw/rest-of", 
                    path_input_data_image = Path("../data/raw/image"))



In [ ]:


# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)


bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), 
                                    climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None) 
vhc_sector = get_preprocessing_data("vehicles", raw_data, 
                                    climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
rest_sector = rest_of_preprocessing(raw_data, 
                    image_scenario_directory = climate_policy_scenario_dir)

# needed for electricity model
scen = scenario_name.split("_")[0]
variant = "_".join(scenario_name.split("_")[1:])

# TODO fix this for real in the future
prep_data_vhc = vhc_sector.prep_data
prep_data_el = get_preprocessing_data_gen(path_image_materials, scen, variant)
_, prep_data_grid = get_preprocessing_data_grid(path_image_materials, scen, variant)


vhc_sector = Sector('vehicles', prep_data_vhc)
rest_sector = Sector(name='rest_of', 
                    data = rest_sector,)
sec_electr_gen = Sector("generation", prep_data_el)
sec_electr_grid = Sector("grid", prep_data_grid)

factory = ModelFactory(
[bld_sector, vhc_sector, rest_sector, sec_electr_gen, sec_electr_grid], complete_timeline
).add(GenericStocks, ["buildings", "vehicles", "generation", "grid"]
).add(GenericMaterials,  "vehicles"
).add(MaterialIntensities, ["buildings", "generation", "grid"]
).add(RestOf, "rest_of", input_sources={
    "gompertz_coefs": "rest_of",
    "gdp_per_capita": "rest_of",
    "population": "rest_of",
    "historic_diff_consumption_mean": "rest_of"
}
)
model = factory.finish()

import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)



Path to image output: C:\Coding\image-materials\data\raw\SSP2_M_CP\EnergyServices
gcap_stock to xarray Dataset
gcap_types_materials to xarray Dataset
gcap_lifetime_distr not in conversion_table
grid_stock_lines to xarray Dataset
materials_grid_kgperkm to xarray Dataset
lifetime_grid_distr not in conversion_table
grid_stock_add to xarray Dataset
materials_grid_add_kgperunit to xarray Dataset
lifetime_grid_distr not in conversion_table


In [6]:
model.grid["inflow_materials"].to_array()

<xarray.DataArray (time: 101, Region: 26, Type: 6, material: 9)> Size: 1MB
<Quantity([[[[2.61038949e+06 0.00000000e+00 9.74061131e+06 ... 1.57233443e-01
    2.37422487e+04 1.15189215e+06]
   [1.22997483e+05 0.00000000e+00 1.60366939e+08 ... 0.00000000e+00
    0.00000000e+00 7.32537865e+07]
   [7.88188217e+07 0.00000000e+00 3.05519211e+07 ... 0.00000000e+00
    0.00000000e+00 2.43901891e+06]
   [5.45570020e+06 0.00000000e+00 1.12965086e+07 ... 0.00000000e+00
    0.00000000e+00 3.08086599e+07]
   [1.40481915e+06 0.00000000e+00 1.45310695e+08 ... 0.00000000e+00
    0.00000000e+00 2.07634101e+06]
   [2.92648173e+04 0.00000000e+00 6.52549682e+07 ... 1.23051585e+02
    0.00000000e+00 3.15767378e+07]]

  [[1.60724854e+07 0.00000000e+00 5.99741277e+07 ... 9.68105417e-01
    1.46183911e+05 7.09233995e+06]
   [7.57310451e+05 0.00000000e+00 9.87398737e+08 ... 0.00000000e+00
    0.00000000e+00 4.51032467e+08]
   [4.36347793e+08 0.00000000e+00 1.69138070e+08 ... 0.00000000e+00
    0.00000000e+00 1.35026190e+07]
   [3.02032267e+07 0.00000000e+00 6.25384459e+07 ... 0.00000000e+00
...
    0.00000000e+00 5.70637367e+06]
   [1.45001943e+07 0.00000000e+00 3.00239318e+07 ... 0.00000000e+00
    0.00000000e+00 8.18834503e+07]
   [3.47803055e+07 0.00000000e+00 3.59758077e+09 ... 0.00000000e+00
    0.00000000e+00 5.14057447e+07]
   [8.23070575e+05 0.00000000e+00 1.83529061e+09 ... 0.00000000e+00
    0.00000000e+00 8.88093151e+08]]

  [[0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
    0.00000000e+00 0.00000000e+00]
   [6.80188071e+05 0.00000000e+00 8.86844808e+08 ... 0.00000000e+00
    0.00000000e+00 4.05100581e+08]
   [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
    0.00000000e+00 0.00000000e+00]
   [3.26050655e+07 0.00000000e+00 6.75116651e+07 ... 0.00000000e+00
    0.00000000e+00 1.84122723e+08]
   [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
    0.00000000e+00 0.00000000e+00]
   [3.58274282e+05 0.00000000e+00 7.98883406e+08 ... 0.00000000e+00
    0.00000000e+00 3.86577950e+08]]]], 'kilogram')>
Coordinates:
  * time      (time) int64 808B 1960 1961 1962 1963 1964 ... 2057 2058 2059 2060
  * Region    (Region) <U13 1kB 'Brazil' 'C.Europe' ... 'W.Africa' 'W.Europe'
  * Type      (Type) <U17 408B 'HV - Substations' ... 'MV - Transformers'
  * material  (material) <U9 324B 'Aluminium' 'Co' ... 'Plastics' 'Steel'

In [ ]:
# sum inflow materials for steel, sum also per types keep regions and year

materials_dict_metal = {
    'Steel' : 'Steel',
    'Aluminium' : 'Aluminium',
    'Copper' : 'Cu',
}

materials_dict_nmm = {
    'Cement' : 'Cement',
    'Sand' : 'Sand'
}

# Conversion factors
# always taking the lower range numbers to be cautios

# https://civiltoday.com/civil-engineering-materials/cement/10-cement-ingredients-with-functions
# Cement: Lime 60-75%, Silica 17-25%, other aggregates
# https://concretesupplyco.com/concrete-basics/
# Concrete:  10% cement, 20% air and water, 30% sand, and 40% gravel --> 30% + 40% = 70%
# https://samsa.org.uk/key_uses/glass.php, https://www.carmeuse.com/na-en/references/case-studies-success-stories/limestone-glassmaking-what-you-need-know
# Sand (silica) in glass: 70%, lime: 15%

from imagematerials.rest_of.preprocessing.sum_sectors_image_materials import sum_inflows_for_output

total_material_dict_metals = sum_inflows_for_output(model, materials_dict_metal, 'metals', save = True)
total_material_dict_nmm = sum_inflows_for_output(model, materials_dict_nmm, 'nmm', save = True)



: 

c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


done cement
Sand
done sand_gravel_crushed_rock


c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


In [17]:
inflow_materials = model.grid["inflow_materials"].to_array().sel(material='Cu').sum(['Type'])
inflow_materials_vehicles = model.vehicles["inflow_materials"].to_array().sel(material='Cu').sum(['Type'])

inflow_materials_vehicles


Magnitude,[[503053918.788158 3510468510.7393847 66995459.17247633 ... 231455552.5790785 13345581.05996393 9315935.988408279] [9962411.716013337 69446838.74943331 1320032.7604785562 ... 4616646.886997723 254204.92899478268 169509.33613866108] [13375250.754253168 93132564.20705421 1753087.248070301 ... 6204782.403167041 328019.700207564 206296.23746585776] ... [507213671.51279444 2229032406.805853 178635110.16041374 ... 425995453.6486685 423830560.72282785 184548539.68546668] [504759816.47139925 2230785775.812096 184331769.06512702 ... 426850588.8876013 440472704.65088475 196127690.65363574] [501974791.6300848 2220901187.974303 189606696.7630388 ... 426743657.74797225 458255992.86867875 208445635.2422587]]
Units,kilogram
